# NASA CMAPSS — Exploratory Data Analysis

This notebook is used to understand the data before the training pipeline runs.

The focus here is on:

- dataset structure and engine counts
- missing values and constant sensors
- engine lifetime and RUL distributions
- sensor behaviour over an engine's lifetime
- relationships between sensors and RUL
- the effect of the project's feature-engineering steps

Model training, tuning, inference and final evaluation are handled by the Python pipeline in `src/`. This notebook is mainly for exploration and documenting the reasoning behind the features used by the model.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

# Make the project root available when the notebook is run from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if not (PROJECT_ROOT / "configs" / "config.yaml").exists():
    raise FileNotFoundError(
        "Project root was not found. Run this notebook from the project root "
        "or from the notebooks/ directory."
    )

sys.path.insert(0, str(PROJECT_ROOT))

from src.data.ingestion import load_training_data
from src.features.feature_engineering import (
    remove_constant_columns,
    create_operating_condition,
    add_lag_features,
    add_rolling_features,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:.3f}")

plt.rcParams["figure.figsize"] = (10, 5)

print(f"Project root: {PROJECT_ROOT}")

## 2. Load the training data

In [ ]:
df = load_training_data()

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

df.head()

The training data contains four CMAPSS subsets (`FD001`–`FD004`). Each record belongs to an engine and a cycle. The target used by the project is `RUL`, calculated from the last observed cycle of each engine.

### Dataset composition

In [ ]:
dataset_summary = (
    df.groupby("dataset_id")
    .agg(
        Engines=("unit_id", "nunique"),
        Records=("unit_id", "size"),
        Max_Cycle=("cycle", "max"),
    )
    .reset_index()
)

dataset_summary

## 3. Basic data checks

In [ ]:
print("Data types:")
display(df.dtypes.to_frame("dtype"))

print("Missing values:")
missing = (
    df.isna()
    .sum()
    .rename("missing_values")
    .to_frame()
)

missing["missing_percentage"] = missing["missing_values"] / len(df) * 100
missing = missing[missing["missing_values"] > 0].sort_values(
    "missing_values", ascending=False
)

if missing.empty:
    print("No missing values found.")
else:
    display(missing)

In [ ]:
constant_columns = [
    column for column in df.columns
    if df[column].nunique(dropna=False) <= 1
]

print(f"Constant columns: {len(constant_columns)}")
constant_columns

Constant sensors do not provide useful variation to a regression model, which is why the production feature-engineering pipeline removes them before creating model features.

## 4. Engine lifetime

In [ ]:
engine_lifetime = (
    df.groupby(["dataset_id", "unit_id"])["cycle"]
    .max()
    .reset_index(name="lifetime")
)

engine_lifetime.describe()

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(engine_lifetime["lifetime"], bins=30)
plt.title("Engine Lifetime Distribution")
plt.xlabel("Lifetime (cycles)")
plt.ylabel("Number of engines")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

The lifetime distribution is useful because engines do not all run for the same number of cycles. This is also the basis for constructing the RUL target.

## 5. RUL distribution

In [ ]:
df["RUL"].describe()

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(df["RUL"], bins=40)
plt.title("RUL Distribution")
plt.xlabel("RUL (cycles)")
plt.ylabel("Number of observations")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

Because RUL is calculated from each engine's final observed cycle, early observations generally have larger RUL values while observations close to the end of an engine's life approach zero.

## 6. Sensor behaviour over an engine lifetime

In [ ]:
dataset_id = "FD001"
unit_id = 1

engine = df[
    (df["dataset_id"] == dataset_id)
    & (df["unit_id"] == unit_id)
].sort_values("cycle")

print(f"{dataset_id} — Engine {unit_id}")
print(f"Cycles: {engine['cycle'].min()} to {engine['cycle'].max()}")
engine[["cycle", "RUL"]].head()

In [ ]:
sensor = "sensor_11"

plt.figure(figsize=(10, 5))
plt.plot(engine["cycle"], engine[sensor])
plt.title(f"{sensor} over Engine Lifetime — {dataset_id}, Engine {unit_id}")
plt.xlabel("Cycle")
plt.ylabel(sensor)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

A single sensor plot is more useful here than producing 21 nearly identical charts. If a sensor shows a consistent change over cycles, it may contain useful degradation information.

## 7. Sensor correlation with RUL

In [ ]:
sensor_columns = [
    column for column in df.columns
    if column.startswith("sensor_")
    and "_lag" not in column
    and "_rolling" not in column
]

correlation = (
    df[sensor_columns + ["RUL"]]
    .corr()["RUL"]
    .drop("RUL")
    .to_frame("Correlation")
)

correlation["Absolute_Correlation"] = correlation["Correlation"].abs()

correlation.sort_values(
    "Absolute_Correlation",
    ascending=False,
).head(15)

In [ ]:
top_corr = correlation.sort_values(
    "Absolute_Correlation",
    ascending=False,
).head(10).sort_values("Correlation")

plt.figure(figsize=(9, 5))
plt.barh(top_corr.index, top_corr["Correlation"])
plt.title("Top Sensor Correlations with RUL")
plt.xlabel("Pearson Correlation")
plt.ylabel("Sensor")
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

Correlation is only a first check. A low linear correlation does not necessarily mean that a sensor is useless: degradation can be nonlinear, depend on operating conditions, or become more informative when its recent history is included. This is one reason the project uses lag and rolling features instead of relying only on raw sensor values.

## 8. Feature engineering preview

In [ ]:
feature_df = df.copy()

feature_df = remove_constant_columns(feature_df)
feature_df = create_operating_condition(feature_df)
feature_df = add_lag_features(feature_df)
feature_df = add_rolling_features(feature_df)

print(f"Before feature engineering: {df.shape}")
print(f"After feature engineering:  {feature_df.shape}")

feature_df.head()

The same feature-engineering functions used by the training/inference pipeline are called here. This avoids maintaining a separate copy of the transformation logic in the notebook.

The main additions are:

- `operating_condition`
- one-cycle lag features
- rolling mean features

The production pipeline later removes rows with missing lag/rolling values before training.

## 9. Quick checks after feature engineering

In [ ]:
new_features = sorted(set(feature_df.columns) - set(df.columns))

print(f"New features created: {len(new_features)}")
print(new_features[:20])

print("\nMissing values introduced by time-series features:")
feature_missing = (
    feature_df[new_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

feature_missing[feature_missing > 0].head(10)

The missing values at the beginning of each engine are expected: a lag feature needs a previous cycle and a rolling window needs enough observations. The training pipeline handles these rows explicitly before model fitting.

## 10. Takeaways

Before modelling, the data shows why this is a time-series regression problem rather than a standard tabular regression task:

1. Engines have different operating lifetimes.
2. Sensor behaviour changes over engine cycles.
3. RUL is tied to the position of an engine within its lifetime.
4. Sensor-to-RUL relationships are not necessarily linear.
5. Recent sensor history can provide additional information beyond the current reading.
6. The same feature-engineering code should be reused during training and inference.

The actual model comparison, Optuna tuning, final training and unseen-test evaluation are kept in the project's Python pipeline so that the notebook remains focused on data understanding.